# 🔬 Notebook 3: Google Search — Deep Dive: Crawler, Ranking, Sharding

## 🛠️ Setup

```bash
cd 06-system-designs/google-search
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Deep dive 1 — the crawler

Key problems:
1. **Discovery**: where do we find URLs? → seeds + links extracted from crawled pages.
2. **Politeness**: don't hammer one host. Obey `robots.txt` and rate-limit per host.
3. **Dedup**: the same URL will show up thousands of times.
4. **Freshness**: some pages change hourly, others never. Use change-history to tune recrawl.

Frontier = priority queue keyed by (host, priority_score). One worker picks the next URL
that is *not* currently being fetched for that host.

In [ ]:
# Toy "frontier" that respects per-host politeness
from collections import deque, defaultdict
import time

class Frontier:
    def __init__(self, min_gap_s=0.0):
        self.q = deque()
        self.seen = set()
        self.last_fetch: dict[str, float] = defaultdict(float)
        self.min_gap_s = min_gap_s

    def add(self, url, host):
        if url in self.seen: return
        self.seen.add(url)
        self.q.append((url, host))

    def next_ready(self):
        now = time.time()
        for i, (url, host) in enumerate(self.q):
            if now - self.last_fetch[host] >= self.min_gap_s:
                self.q.remove((url, host))
                self.last_fetch[host] = now
                return url, host
        return None

f = Frontier(min_gap_s=0.01)
for i in range(5): f.add(f"https://a.com/{i}", "a.com")
for i in range(3): f.add(f"https://b.com/{i}", "b.com")

for _ in range(8):
    n = f.next_ready()
    print("fetching:", n)


## Deep dive 2 — ranking

Classic ingredients:
- **TF-IDF / BM25**: how well does the doc match query words?
- **PageRank**: how "important" is the page based on the link graph?
- **Freshness**: decay by age for time-sensitive queries.
- **User signals**: click-through rate, dwell time, pogo-sticking back to SERP.
- **Modern**: learned rankers — a gradient-boosted tree or neural model over hundreds of features.

Final score = weighted combination; weights are learned from click logs.

In [ ]:
# Mini PageRank on a 5-node graph
def pagerank(links: dict[str, list[str]], d=0.85, iters=30) -> dict[str, float]:
    nodes = set(links) | {b for out in links.values() for b in out}
    pr = {n: 1/len(nodes) for n in nodes}
    for _ in range(iters):
        new = {n: (1-d)/len(nodes) for n in nodes}
        for src, outs in links.items():
            if not outs: continue
            share = pr[src] / len(outs)
            for dst in outs:
                new[dst] += d * share
        pr = new
    return pr

links = {"A":["B","C"], "B":["C"], "C":["A"], "D":["C"], "E":["D","C"]}
for n, s in sorted(pagerank(links).items(), key=lambda x: -x[1]):
    print(f"{n}: {s:.3f}")


## Deep dive 3 — sharding the index

You **cannot** keep the whole inverted index on one machine. Two sharding strategies:

| Strategy | Query fan-out | Failure blast radius |
|---|---|---|
| **By term** (each shard owns a slice of vocabulary) | Low: only shards for query terms | A down shard loses some queries entirely |
| **By document** (each shard owns a doc-id range) | High: ask all shards | A down shard loses a slice of docs (graceful) |

Google uses **document sharding**. Query all shards, merge top-K.
With 1000 shards, network cost is dominated by the merge step.
